<hr>

**<font color=skyblue>示範 PyTorch 的淺度機器學習（典型的 Neural Network）應用在AT&T 人臉辨識的訓練與測試過程</font>**

參考講義：https://ntpuccw.blog/sml-lesson-9-%e5%be%9e-pytorch-%e7%9a%84%e6%b7%ba%e5%ba%a6%e5%88%b0%e6%b7%b1%e5%ba%a6%e6%a9%9f%e5%99%a8%e5%ad%b8%e7%bf%92/

<hr>

<font color=yellow>Import packages</font>

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

*<font color=yellow>Load data and prepare for PyTorch Tensor</font>*

Tensors are used in machine learning to represent data and model parameters. They are used to encode the inputs and outputs of a model as well as the model’s parameters. In PyTorch, tensors are used to represent the inputs and outputs of a model as well as the model’s parameters.

Torch is a scientific computing framework with wide support for machine learning algorithms that puts GPUs first.

In [3]:
df = pd.read_csv('data/face_data.csv')
n_persons = df['target'].nunique() 
X = np.array(df.drop('target', axis=1)) # 400 x 4096
y = np.array(df['target'])

test_size = 0.3
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size) # deafult test_size=0.25

# prepare data for PyTorch Tensor
X_train = torch.from_numpy(X_train).float() # convert to float tensor
y_train = torch.from_numpy(y_train).float() # 
train_dataset = TensorDataset(X_train, y_train) # create your datset
X_test = torch.from_numpy(X_test).float()
y_test = torch.from_numpy(y_test).float()
test_dataset = TensorDataset(X_test, y_test) # create your datset

# create dataloader for PyTorch
batch_size = 32 # 32, 64, 128, 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True) # convert to dataloader
test_loader = DataLoader(test_dataset, batch_size=len(X_test), shuffle=False)

print("Data preparation done.")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("Batch size:", batch_size)
print("Test size:", len(X_test))
print("Number of batches in train_loader:", len(train_loader))
print("Number of batches in test_loader:", len(test_loader))

Data preparation done.
X_train shape: torch.Size([280, 4096])
y_train shape: torch.Size([280])
X_test shape: torch.Size([120, 4096])
y_test shape: torch.Size([120])
Batch size: 32
Test size: 120
Number of batches in train_loader: 9
Number of batches in test_loader: 1


<hr>

*<font color=yellow>Set up NN  model</font>* 

下圖的結構等同於 sklearn 的 MLPClassifier 的 hidden_layer_sizes=(512, 128)，輸出層的設定為 40。

![My image](pictures/Copilot_20260428_164229.png)

<hr>

In [4]:
# select device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = "cpu" # run faster than cuda in some cases
print("Using {} device".format(device))

# Create a neural network
class MLP(nn.Module):
    def __init__(self): # constructor
        # call the parent constructor
        super(MLP, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(64*64, 512), # image length 64x64=4096,  fully connected layer
            nn.ReLU(), # you may want to take ReLU out to see what happen
            nn.Linear(512, 128), # second hidden layer
            nn.ReLU(),
            nn.Linear(128, 40) # 40 classes,  fully connected layer
            # nn.Softmax()
        )
    # Specify how data will pass through this model
    def forward(self, x):
        # out = self.mlp(x) 

        # Apply softmax to x here~
        x = self.mlp(x) # pass through the model
        out = F.log_softmax(x, dim=1) # it’s faster and has better numerical propertie than softmax
        # out = F.softmax(x, dim=1)
        return out

# define model, optimizer, loss function
model = MLP().to(device) # start an instance
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # default lreaning rate=1e-3
loss_fun = nn.CrossEntropyLoss() # define loss function

print(model)

Using cuda device
MLP(
  (mlp): Sequential(
    (0): Linear(in_features=4096, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=40, bias=True)
  )
)


<hr>

*<font color=yellow>Test the forward process of the nn model</font>*

逐步觀察資料進入每一層後的變化，並且在每一層之後加入 ReLU 激活函數，最後在輸出層使用 softmax 函數來獲得機率分布。

In [10]:
input = torch.randn(1, 64 * 64) # test input: 1 x 4096
m1 = nn.Linear(64*64, 512)

# 通過第一層
print('Passing through the first layer...')
output = m1(input)
# print(output.size())
output = F.relu(output)
print(output.size())

# 通過第二層
print('Passing through the second layer...')
m2 = nn.Linear(512, 128)
output = m2(output)
# print(output.size())
output = F.relu(output)
print(output.size())
# 通過第三層
print('Passing through the third layer (output layer)...')
m3 = nn.Linear(128, 40)
output = m3(output)
# output = F.log_softmax(output, dim=1)
output = F.softmax(output, dim=1) # it’s faster and has better numerical propertie than softmax
print(output.size())
print(output)
print("The predicted class is:", output.argmax(dim=1)) # get the index of the max log-probability

Passing through the first layer...
torch.Size([1, 512])
Passing through the second layer...
torch.Size([1, 128])
Passing through the third layer (output layer)...
torch.Size([1, 40])
tensor([[0.0246, 0.0239, 0.0267, 0.0173, 0.0242, 0.0224, 0.0203, 0.0277, 0.0355,
         0.0272, 0.0230, 0.0250, 0.0248, 0.0236, 0.0291, 0.0256, 0.0266, 0.0196,
         0.0240, 0.0256, 0.0235, 0.0238, 0.0251, 0.0302, 0.0242, 0.0236, 0.0240,
         0.0228, 0.0274, 0.0263, 0.0223, 0.0264, 0.0217, 0.0250, 0.0206, 0.0268,
         0.0300, 0.0239, 0.0299, 0.0260]], grad_fn=<SoftmaxBackward0>)
The predicted class is: tensor([8])


使用複合指令

In [11]:
output = nn.Linear(64*64, 512)(X_train)
print(output.shape)
pr = nn.ReLU()(output)
p = nn.Linear(512, 40)(pr)
print(p.shape)

torch.Size([280, 512])
torch.Size([280, 40])


<hr>
*<font color=yellow>Training</font>*

典型的深度學習訓練過程：設定 epochs（訓練迭代次數）迴圈，在每一個 epoch 中，對於每一個批次（batch）資料：

- 從 Train Dataloader 提取批次資料
- 將該批次資料傳入模型，獲得預測結果
- 與真實標籤計算損失（loss）
- 反向傳播（backpropagation）以計算梯度
- 更新模型參數（optimizer step）

<font color=yellow>可以在迭代過程中觀察損失值的變化及準確率（accuracy），以確保模型正在學習。</font>
<hr>

In [13]:
from tqdm import tqdm

epochs = 100 # Repeat the whole dataset epochs times
model.train() # Sets the module in training mode. The training model allow the parameters to be updated during backpropagation.
# for epoch in range(epochs):
for epoch in tqdm(range(epochs)):
    trainAcc = 0
    samples = 0
    losses = []
    for batch_num, input_data in enumerate(train_loader):
    # for batch_num, input_data in tqdm(enumerate(train_loader), total=len(train_loader)):
        
        x, y = input_data
        x = x.to(device).float()
        y = y.to(device)

        # perform training based on the backpropagation
        y_pre = model(x) # predict y
        loss = loss_fun(y_pre, y.long()) # the loss function nn.CrossEntropyLoss()
        losses.append(loss.item())

        optimizer.zero_grad() # Zeros the gradients accumulated from the previous batch/step of the model
        loss.backward() # Performs backpropagation and calculates the gradients
        optimizer.step() # Updates the weights in our neural network based on the results of backpropagation
        
        # Record the training accuracy for each batch
        trainAcc += (y_pre.argmax(dim=1) == y).sum().item() # comparison
        samples += y.size(0)
        # if batch_num % 4 == 0:
        #     print('\tEpoch %d | Batch %d | Loss %6.2f' % (epoch, batch_num, loss.item()))
    # print('Epoch %d | Loss %6.2f | train accuracy %.4f' % (epoch, sum(losses)/len(losses), trainAcc/samples))

print('Finished ... Loss %7.4f | train accuracy %.4f' % (sum(losses)/len(losses), trainAcc/samples))

100%|██████████| 100/100 [00:04<00:00, 21.50it/s]

Finished ... Loss  0.0034 | train accuracy 1.0000


<hr>

*<font color=yellow>Testing (1)</font>* Compute test accuracy on the test set.

測試過程與訓練過程類似，但不需要反向傳播和參數更新。通常在測試過程中，我們會將模型設置為評估模式（model.eval()）。



In [15]:
model.eval() # Sets the module in evaluation mode. The evaluation model does not allow the parameters to be updated during backpropagation.
# test the model
testAcc = 0
samples = 0
with torch.no_grad():
    for x, y_truth in test_loader:
        x = x.to(device).float()
        y_truth = y_truth.to(device)
        y_pre = model(x).argmax(dim=1) # the predictions for the batch
        testAcc += (y_pre == y_truth).sum().item() # comparison
        samples += y_truth.size(0)

    print('Test Accuracy:{:.3f}'.format(testAcc/samples))


Test Accuracy:0.883


*<font color=yellow>Testing (2)</font>*

- 計算整體測試準確率（accuracy）
- 記錄每一筆測試資料的預測結果，並與真實標籤一起紀錄在 EXCEL 檔。

In [17]:
import csv

# use eval() in conjunction with a torch.no_grad() context, 
# meaning that gradient computation is turned off in evaluation mode
model.eval() 
testAcc = 0
samples = 0

with open('data/mlp_att.csv', 'w') as f:
    fieldnames = ['ImageId', 'Prediction', 'Ground_Truth']
    writer = csv.DictWriter(f, fieldnames=fieldnames, lineterminator = '\n')
    writer.writeheader()
    image_id = 1

    with torch.no_grad():
        for x, y_truth in test_loader:
            x = x.to(device).float()
            y_truth = y_truth.to(device).long()
            yIdx = 0
            y_pre = model(x).argmax(dim=1) # the predictions for the batch
            testAcc += (y_pre == y_truth).sum().item() # comparison
            samples += y_truth.size(0)
            for y in y_pre:
                writer.writerow({fieldnames[0]: image_id,fieldnames[1]: y.item(), fieldnames[2]: y_truth[yIdx].item()})
                image_id += 1
                yIdx += 1

        print('Test Accuracy:{:.3f}'.format(testAcc/samples))

Test Accuracy:0.883
